# Detectability of Gravitational Wave

The detectability is expressed as segnal noise ratio (snr). Tipically for a gravitational wave to be detected we need a value of `snr` > 12. The result is stored in `det` variable as 0 1.

In [1]:
import h5py as h5py

path = "/Users/ffara/Desktop/LML/astrostatistics_bicocca_2025/exercise/sample_2e7_design_precessing_higherordermodes_3detectors.h5"
with h5py.File(path, 'r') as f:
    fin = {key: f[key][()] for key in f.keys()}

In [2]:
N_lim = 100000 

data = {k: v[:N_lim] for k, v in fin.items()}
len(data['q'])

100000

In [3]:
print(f.keys)

<bound method MappingHDF5.keys of <Closed HDF5 file>>


In [4]:
import numpy as np
import matplotlib.pyplot as plt

Y = data['det']

X = np.column_stack([
    data['q'],
    data['mtot'],
    # data['iota'],
    # data['ra'],
    # data['dec'],
    data['psi'],
    data['z'],
    # data['snr'],
])

In [5]:
np.bincount(Y), np.bincount(data['snr'] > 12)

(array([85618, 14382]), array([85618, 14382]))

In [6]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import  confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

### Random Forest

To debug and prepare, I've been drove by physical motivation: snr should be in primis proportional to the distance and redshift. 

In [8]:
help(RandomForestClassifier)

Help on class RandomForestClassifier in module sklearn.ensemble._forest:

class RandomForestClassifier(ForestClassifier)
 |  RandomForestClassifier(n_estimators=100, *, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features='sqrt', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None, monotonic_cst=None)
 |  
 |  A random forest classifier.
 |  
 |  A random forest is a meta estimator that fits a number of decision tree
 |  classifiers on various sub-samples of the dataset and uses averaging to
 |  improve the predictive accuracy and control over-fitting.
 |  Trees in the forest use the best split strategy, i.e. equivalent to passing
 |  `splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
 |  The sub-sample size is controlled with the `max_samples` paramet

In [9]:
gscv_rf = GridSearchCV(
    estimator=RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced_subsample"),
    param_grid={
        'max_depth': [3, 4, 5, 6, 7, 8],
    },
    cv=5,
    n_jobs=-1,
    scoring='roc_auc'
)

cv = make_pipeline(StandardScaler(), gscv_rf)

cv.fit(X_train, Y_train)
print("Best parameters found: ", gscv_rf.best_params_)
print("CV set score: ", gscv_rf.score(X_train, Y_train))

Best parameters found:  {'max_depth': 8}
CV set score:  0.5554187874011471


In [10]:
rf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=300, max_depth=gscv_rf.best_params_['max_depth'], random_state=42, class_weight="balanced_subsample"))
rf.fit(X_train, Y_train)
print("Train set score Random Forest: ", rf.score(X_train, Y_train))
print("Test set score Random Forest: ", rf.score(X_test, Y_test))

Train set score Random Forest:  0.92912
Test set score Random Forest:  0.9266


In [11]:
cm_rf = confusion_matrix(Y_test, rf.predict(X_test))
print("Confusion Matrix for Random Forest:")
print(cm_rf)

Confusion Matrix for Random Forest:
[[19755  1643]
 [  192  3410]]


In [12]:
completeness = cm_rf[1, 1] / (cm_rf[1, 0] + cm_rf[1, 1])
contamination = cm_rf[0, 1] / (cm_rf[0, 1] + cm_rf[1, 1])

print(f"Completeness (True Positives / Total Positives): {completeness:.4f}")
print(f"Contamination (False Positives / Total Estimated Positives): {contamination:.4f}")


Completeness (True Positives / Total Positives): 0.9467
Contamination (False Positives / Total Estimated Positives): 0.3252


### Boosting

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

In [14]:
help(GradientBoostingClassifier)

Help on class GradientBoostingClassifier in module sklearn.ensemble._gb:

class GradientBoostingClassifier(sklearn.base.ClassifierMixin, BaseGradientBoosting)
 |  GradientBoostingClassifier(*, loss='log_loss', learning_rate=0.1, n_estimators=100, subsample=1.0, criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, verbose=0, max_leaf_nodes=None, warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)
 |  
 |  Gradient Boosting for classification.
 |  
 |  This algorithm builds an additive model in a forward stage-wise fashion; it
 |  allows for the optimization of arbitrary differentiable loss functions. In
 |  each stage ``n_classes_`` regression trees are fit on the negative gradient
 |  of the loss function, e.g. binary or multiclass log loss. Binary
 |  classification is a special case where only a single regression

In [15]:
gscv_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(n_estimators=200, random_state=42),
    param_grid={
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 4, 5, 6, 7, 8]
    },
    cv=5,
    n_jobs=-1,
    scoring='roc_auc'
)

cv = make_pipeline(StandardScaler(), gscv_gb)
cv.fit(X_train, Y_train)
print("Best parameters found: ", gscv_gb.best_params_)
print("CV set score: ", gscv_gb.score(X_train, Y_train))

Best parameters found:  {'learning_rate': 0.1, 'max_depth': 3}
CV set score:  0.5005882420066919


In [16]:
gb = make_pipeline(StandardScaler(), GradientBoostingClassifier(n_estimators=300, learning_rate=gscv_gb.best_params_['learning_rate'], max_depth=gscv_gb.best_params_['max_depth'], random_state=42))

gb.fit(X_train, Y_train)
print("Train set score Gradient Boosting: ", gb.score(X_train, Y_train))
print("Test set score Gradient Boosting: ", gb.score(X_test, Y_test))

Train set score Gradient Boosting:  0.9554266666666666
Test set score Gradient Boosting:  0.95172


In [17]:
cm = confusion_matrix(Y_test, rf.predict(X_test))
print("Confusion Matrix for Gradient Boosting:")
print(cm)

Confusion Matrix for Gradient Boosting:
[[19755  1643]
 [  192  3410]]


In [18]:
completeness = cm[1, 1] / (cm[1, 0] + cm[1, 1])
contamination = cm[0, 1] / (cm[0, 1] + cm[1, 1])

print(f"Completeness (True Positives / Total Positives): {completeness:.4f}")
print(f"Contamination (False Positives / Total Estimated Positives): {contamination:.4f}")

Completeness (True Positives / Total Positives): 0.9467
Contamination (False Positives / Total Estimated Positives): 0.3252
